# Certified Numerical Analysis with Rocq and AI Assistance

If you are using Google Colab, first enable a GPU runtime. Then run the setup cell: it installs the workshop package, downloads the retrieval cache, and creates the retrieval/LLM clients used later.


## 0. Grab a GPU

Google Colab only.

Before doing anything, try to get a GPU instance. A T4 runtime is enough for the local retrieval embeddings used in the notebook.


Go to instance options.

![step_0](https://raw.githubusercontent.com/theostos/integral-tp/refs/heads/main/img/step_0_option.png)

Change runtime.

![step_1](https://raw.githubusercontent.com/theostos/integral-tp/refs/heads/main/img/step_1_change_runtime.png)

Add a T4 GPU.

![step_2](https://raw.githubusercontent.com/theostos/integral-tp/refs/heads/main/img/step_2_t4.png)

Then connect to the GPU instance.

![step_3](https://raw.githubusercontent.com/theostos/integral-tp/refs/heads/main/img/step_3_connect.png)


In [ ]:
%pip install -q integral-tp[colab]@git+https://github.com/theostos/integral-tp.git

import os

from huggingface_hub import hf_hub_download

# Workshop services. Change these values when the servers run elsewhere.
# For one ngrok URL routed by path, set:
# os.environ["ROCQ_SERVER_URL"] = "https://<ngrok-host>/rocq"
# os.environ["WORKSHOP_LLM_SERVER_URL"] = "https://<ngrok-host>/llm"
os.environ["ROCQ_SERVER_URL"] = "https://crabbily-cataracted-gilma.ngrok-free.dev/rocq"
os.environ["WORKSHOP_LLM_SERVER_URL"] = "https://crabbily-cataracted-gilma.ngrok-free.dev/llm"
# os.environ.setdefault("ROCQ_SERVER_HOST", "127.0.0.1")
# os.environ.setdefault("ROCQ_SERVER_PORT", "5000")
# os.environ.setdefault("WORKSHOP_LLM_SERVER_URL", "http://127.0.0.1:8010")
# os.environ.setdefault("WORKSHOP_LLM_SERVER_POLL_SECONDS", "2.0")

import workshop_api
from workshop_api import (
    LLMClient,
    RetrievalClient,
    RetrievalExplorer,
    download_retrieval_cache,
    format_retrieval_hits,
)

zip_path = hf_hub_download(
    repo_id="theostos/integral-tp-retrieval-cache",
    filename="retrieval_cache.zip",
    repo_type="dataset",
)

cache_path = download_retrieval_cache(zip_path, cache_dir="/content/rocq-doc-cache")
retriever = RetrievalClient.from_env(cache_dir=cache_path)
remote_llm = LLMClient.from_env()

_ = retriever.search("warmup", library="Coquelicot", k=3)

# Small helpers for fallback context cells.
def expected_hit(name, *, library=None, kind=None, query=None, k=12):
    """Return a retrieval hit by exact name, preferring the local cache metadata."""
    def same(value, expected):
        return expected is None or str(value).lower() == str(expected).lower()

    local = retriever._local_retriever()
    for item in local.metadata:
        if (
            item.get("name") == name
            and same(item.get("library"), library)
            and same(item.get("kind"), kind)
        ):
            hit = dict(item)
            hit.setdefault("score", 1.0)
            return hit

    hits = retriever.search(query or name, library=library, kind=kind, k=k)
    for hit in hits:
        if hit.get("name") == name:
            return hit
    names = [hit.get("name") for hit in hits]
    raise ValueError(f"Could not find retrieval hit {name!r}. Closest hits: {names}")


def set_hits(target, *hits):
    """Replace a selected-hit list and show the loaded names."""
    target.clear()
    for hit in hits:
        if hasattr(hit, "as_retrieval_hit"):
            hit = hit.as_retrieval_hit()
        target.append(hit)
    print([hit.get("name") for hit in target])
    return target


def show_usage(label, results):
    """Print total LLM usage for one strategy or proof."""
    if not isinstance(results, (list, tuple)):
        results = [results]
    usages = [result.usage for result in results if getattr(result, "usage", None) is not None]
    usage = workshop_api.LLMUsage.aggregate(usages)
    print(f"{label}: {len(results)} request(s), {usage.summary()}")
    return usage



## How this notebook talks to Rocq

Rocq runs in a separate server. In the notebook, Python is only the interface: each call such as `add_definition`, `add_theorem`, or `run_tac` sends a Rocq command to the server and receives either new goals or an error.


The goal is not to write a complete `.v` file by hand. We interact with the prover through a higher-level layer: retrieval helps us find formal ingredients, the LLM proposes proof scripts, and Rocq checks every accepted step.


## Goal of the session

We start from a numerical estimate that is wrong, then ask Rocq for a certified enclosure. After that, we build an analytic closed form for the same integral.

The recurring workflow is simple: describe the formal ingredient we need, retrieve candidate lemmas or tactics, ask the LLM for the proof plumbing, and let Rocq check the result.


## 1. A numerical disagreement

We study

$$
f(x)=\operatorname{sech}(10x-2)^2
     +\operatorname{sech}(100x-40)^4
     +\operatorname{sech}(1000x-600)^6.
$$

A direct numerical integration in Mathematica gives about `0.209736`.


Sage can also be fragile with the unstable exponential definition of `sech`: it may even return `nan`. With a stable implementation, it agrees with Mathematica and gives about `0.2097360688339336`.

We now ask Rocq for a certified enclosure instead of trusting floating point code.

<details>
<summary>Reference Mathematica call</summary>

```Mathematica
Clear[x, f];
f[x_] := Sech[10 x - 2]^2 + Sech[100 x - 40]^4 + Sech[1000 x - 600]^6;
NIntegrate[f[x], {x, 0, 1}]
```

</details>


We use a first Rocq document only for the certified numerical part. Later we will create a second document for the analytic proof. This is intentional: if we reset or replay a proof while experimenting with the LLM, we do not want to trigger the expensive interval computation again.


`new_document()` creates an independent Rocq session. `add_import(lib, modules)` sends an import command to that session; here `"Coq"`, `"Coquelicot"`, and `"Interval"` are library roots, and the second argument lists modules to import.


In [ ]:
doc_numerical_integral = workshop_api.new_document()

doc_numerical_integral.add_import("Coq", "Reals Lra Psatz")
doc_numerical_integral.add_import("Coquelicot", "Coquelicot")
doc_numerical_integral.add_import("Interval", "Tactic Plot")


`add_definition(command)` sends one complete Rocq `Definition`. The string must include the name, arguments, body, and final period.


In [ ]:
doc_numerical_integral.add_definition("""Definition sech (u : R) : R :=
  2 * exp (u) / (exp (2 * u) + 1).""")

doc_numerical_integral.add_definition("""Definition f (x : R) : R :=
    (sech (10 * x - 2))^2
  + (sech (100 * x - 40))^4
  + (sech (1000 * x - 600))^6.""")

doc_numerical_integral.add_definition("Definition I : R := RInt f 0 1.")


The command below asks Interval to compute a certified decimal enclosure. It is the only heavy numerical cell in the notebook.


`execute(command)` sends a standalone Rocq command. Here we use it only for the heavy interval computation.


In [ ]:
result = doc_numerical_integral.execute("""Do integral
  ltac:(let J := eval cbv [I f sech] in I in exact J)
  with (i_prec 25, i_degree 3, i_fuel 300,
        i_width (-15), i_decimal).""")

print(result['feedback'][0])

On the reference file this prints:

```text
0.21078887 <= I <= 0.21081871
```

This is incompatible with the value printed by Mathematica/Sage. We can package the statement as a theorem. This is the only proof in the session that we run manually from start to finish.


`add_theorem(header)` opens a proof session. `run_tac(tactic)` sends one tactic command to the current proof state, and `qed()` closes the theorem once no goals remain.


In [ ]:
I_digits = doc_numerical_integral.add_theorem("""Theorem I_first_4_decimal_digits :
  Rabs (I - 0.2108) <= 1e-4.""")

I_digits.run_tac("unfold I, f, sech.")
I_digits.run_tac("integral with (i_prec 25, i_degree 3, i_fuel 300).")
I_digits.qed()


## 2. Rocq objects we will manipulate

A few words will appear throughout the notebook:

- a **goal** is what remains to prove;
- a **theorem** or **lemma** is a named statement, reusable after it is proved;
- a **tactic** is a Rocq command that transforms the current goals;
- a **retrieval hit** is a library fact or tactic that we may give to the LLM.

In practice, we will collect a small context of useful hits, then ask the LLM to assemble a proof script from the current proof state.


## 3. Analytic document

We now start a fresh Rocq document. It repeats the definitions, but it does not run the interval computation.


In [ ]:
doc_analytic_integral = workshop_api.new_document()

doc_analytic_integral.add_import("Coq", "Reals Lra Psatz")
doc_analytic_integral.add_import("Coquelicot", "Coquelicot")
doc_analytic_integral.add_import("Interval", "Tactic Plot")


In [ ]:
doc_analytic_integral.add_definition("""Definition sech (u : R) : R :=
  2 * exp (u) / (exp (2 * u) + 1).""")

doc_analytic_integral.add_definition("""Definition f (x : R) : R :=
    (sech (10 * x - 2))^2
  + (sech (100 * x - 40))^4
  + (sech (1000 * x - 600))^6.""")

doc_analytic_integral.add_definition("Definition I : R := RInt f 0 1.")


The analytic candidate uses the exponential presentation of the hyperbolic tangent:

$$
\tanh(u)=\frac{e^{2u}-1}{e^{2u}+1}.
$$

We use the Rocq name `tanh_exp` because Stdlib already defines `tanh` through `sinh` and `cosh`.


For the three powers of `sech`, we use these one-variable candidates:

$$
A_2(u)=\tanh(u),\qquad
A_4(u)=\tanh(u)-\frac{1}{3}\tanh(u)^3,
$$

$$
A_6(u)=\tanh(u)-\frac{2}{3}\tanh(u)^3+\frac{1}{5}\tanh(u)^5.
$$


Then we rescale them to match the three bumps of the integrand:

$$
F_2(x)=\frac{A_2(10x-2)}{10},\qquad
F_4(x)=\frac{A_4(100x-40)}{100},\qquad
F_6(x)=\frac{A_6(1000x-600)}{1000}.
$$

So `F2`, `F4`, and `F6` correspond to the square, fourth-power, and sixth-power terms respectively. The proposed closed form is

$$
F(1)-F(0)\quad\text{where}\quad F=F_2+F_4+F_6.
$$


In [ ]:
doc_analytic_integral.add_definition("""Definition tanh_exp (u : R) : R :=
  (exp (2 * u) - 1) / (exp (2 * u) + 1).""")

doc_analytic_integral.add_definition("""Definition A2 (u : R) : R :=
  tanh_exp u.""")

doc_analytic_integral.add_definition("""Definition A4 (u : R) : R :=
  tanh_exp u - (/ 3) * (tanh_exp u)^3.""")

doc_analytic_integral.add_definition("""Definition A6 (u : R) : R :=
  tanh_exp u - (2 / 3) * (tanh_exp u)^3 + (/ 5) * (tanh_exp u)^5.""")


Now we define the rescaled functions `F2`, `F4`, `F6`, their sum `F`, and the candidate value `I_closed_form`.


In [ ]:
doc_analytic_integral.add_definition("""Definition F2 (x : R) : R :=
  A2 (10 * x - 2) / 10.""")

doc_analytic_integral.add_definition("""Definition F4 (x : R) : R :=
  A4 (100 * x - 40) / 100.""")

doc_analytic_integral.add_definition("""Definition F6 (x : R) : R :=
  A6 (1000 * x - 600) / 1000.""")

doc_analytic_integral.add_definition("""Definition F (x : R) : R :=
  F2 x + F4 x + F6 x.""")

doc_analytic_integral.add_definition("""Definition I_closed_form : R :=
  F 1 - F 0.""")


The formal work now has two phases:

1. prove that `F2`, `F4`, and `F6` differentiate to the three terms of `f`;
2. combine those facts and apply the fundamental theorem of calculus.


## 4. Formalizing `F2_derivative`

First we need to express the informal statement "the derivative of `F2` is `sech(10*x - 2)^2`" in Coquelicot's language.

Search for the predicate used to say that a real function has a derivative at a point.

<details>
<summary>Hint</summary>

Try a query like: `real function has derivative at a point`.

</details>


## Retrieval in one minute

The retrieval system is a semantic search engine over Rocq documentation. Each library item is converted into a vector, called an **embedding**, by the model `Qwen/Qwen3-Embedding-4B`. The query is converted into a vector in the same space.


The notebook compares the query vector with all documentation vectors using cosine similarity: after normalization, this is just a dot product. High cosine means that the texts point in a similar direction, so we show the nearest Rocq definitions, lemmas, or tactics.


The search is only a suggestion mechanism. Rocq does not trust the retriever or the LLM: only the final proof script checked by the Rocq kernel matters.


`RetrievalExplorer(retriever, selected_hits=..., default_query=..., default_library=..., default_kind=..., default_k=...)` opens a small search UI. The explorer appends chosen hits to `selected_hits`; the other arguments only initialize the search fields.


In [ ]:
# This list is the context that we curate during retrieval.
# RetrievalExplorer writes into it, and we can also add already proved objects.
selected_hits = []

selected_hits.clear()

retrieval_explorer = RetrievalExplorer(
    retriever,
    selected_hits=selected_hits,
    default_query="derivative of a function at a point",
    default_kind="definition",
    default_library="Coquelicot",
    default_k=8,
)
retrieval_explorer.display()


Based on the search results, did you find a definition that can express the derivative of `F2`?

<details>
<summary>Expected retrieval hit</summary>

Name: `is_derive`

Signature:

```coq
is_derive : (R -> R) -> R -> R -> Prop
```

</details>

Now fill the following cell with the formal statement.


In [ ]:
f2_derivative = doc_analytic_integral.add_theorem("""Lemma F2_derivative (x : R) :
  [TO FILL].""")

<details>
<summary>Expected answer</summary>

```coq
Lemma F2_derivative (x : R) :
  is_derive F2 x ((sech (10 * x - 2)) ^ 2).
```

</details>


To prove this, we need to show two things:

- `F2` is differentiable at `x`;
- the derivative computed from the definition of `F2` is equal to the right-hand side.

So we are looking for a tactic that helps with differentiability and computes derivatives of real functions.

<details>
<summary>Hint</summary>

Try a query like: `a tactic to obtain automatically the derivative`.

</details>

<details>
<summary>Expected retrieval hit</summary>

Name: `auto_derive`

Signature:

```coq
Ltac auto_derive := ...
```

It is a Coquelicot tactic for derivative and differentiability goals.

</details>


In [ ]:
selected_hits.clear()

retrieval_explorer = RetrievalExplorer(
    retriever,
    selected_hits=selected_hits,
    default_query="",
    default_library="Coquelicot",
    default_kind="Ltac",
    default_k=8,
)
retrieval_explorer.display()


Now we run only the first proof steps by hand. This is useful: `unfold` exposes the definitions, and the derivative tactic exposes the side conditions that are really needed.


In [ ]:
f2_derivative.run_tac("unfold F2, A2, sech, tanh_exp.")
f2_derivative.run_tac("auto_derive.")
print(f2_derivative)


`checkpoint("name")` records the current proof state. Later, `reverse("name")` rolls the proof back to that state; this is how the three LLM strategies start from the same goal.


In [ ]:
# here we do a checkpoint so that later we will be able to
# reset the proof to this more advance point
f2_derivative.checkpoint("after_auto_derive")

Inspect the goals. One side condition says that an expression of the form `exp u + 1` is not zero.

This denominator appears several times, so we prove it once as a separate lemma.

<details>
<summary>Hint</summary>

Search for facts saying: `exp` is positive, a sum of positive real numbers is positive, and a positive real number is not zero.

</details>

<details>
<summary>Expected retrieval hits</summary>

```coq
exp_pos : forall x : R, 0 < exp x
Rplus_lt_0_compat : forall r1 r2 : R, 0 < r1 -> 0 < r2 -> 0 < r1 + r2
Rgt_not_eq : forall r1 r2 : R, r1 > r2 -> r1 <> r2
```

</details>


In [ ]:
selected_hits.clear()

retrieval_explorer = RetrievalExplorer(
    retriever,
    selected_hits=selected_hits,
    default_query="",
    default_library="Stdlib",
    default_kind="start_theorem_proof",
    default_k=8,
)
retrieval_explorer.display()


`set_hits(selected_hits, ...)` clears the selected context and reloads the listed ingredients. It is the fallback path when the explorer was hard to use.

`remote_llm.prove(theorem, selected_hits=..., extra_context=..., tools=..., max_tool_calls=...)` asks the LLM for proof steps and lets Rocq check them. `selected_hits` is the formal context, `extra_context` is informal mathematical guidance, and `tools` enables the agentic loop with `run_tac` and `reverse`.


In [ ]:
set_hits(
    selected_hits,
    expected_hit("Rgt_not_eq", library="Stdlib", kind="start_theorem_proof"),
    expected_hit("Rplus_lt_0_compat", library="Stdlib", kind="start_theorem_proof"),
    expected_hit("exp_pos", library="Stdlib", kind="start_theorem_proof"),
)


In [ ]:
sech_denominator_nonzero = doc_analytic_integral.add_theorem("""Lemma sech_denominator_nonzero (u : R) :
  exp u + 1 <> 0.""")

result = remote_llm.prove(
    sech_denominator_nonzero,
    selected_hits=selected_hits,
    extra_context=(
        "denominator is nonzero because it is strictly "
        "positive: the exponential is positive, and adding 1 preserves "
        "strict positivity."
    ),
    verbose=True,
    tools={
        "run_tac": sech_denominator_nonzero.run_tac,
        "reverse": sech_denominator_nonzero.reverse,
    },
    max_tool_calls=30,
)
print(result)


<details>
<summary>Expected proof</summary>

```coq
apply Rgt_not_eq with (r1 := ((exp u) + 1)) (r2 := 0).
apply Rplus_lt_0_compat with (r1 := exp u) (r2 := 1).
apply exp_pos.
lra.
```

</details>


The lemma is now a local result we proved during the session. We add it to the selected context so the LLM can use it exactly like a library fact.

There is one more technical ingredient in the derivative proofs. After `auto_derive`, the equality goal contains expressions such as `exp (2 * u)`, while `sech` contains `exp u`. We need the theorem saying that the exponential of a sum is a product.

<details>
<summary>Hint</summary>

Try a query like: `exponential of a sum`.

</details>

<details>
<summary>Expected retrieval hit</summary>

Name: `exp_plus`

Signature:

```coq
exp_plus : forall x y : R, exp (x + y) = exp x * exp y
```

</details>


In [ ]:
selected_hits.clear()
selected_hits.append(sech_denominator_nonzero.as_retrieval_hit())

retrieval_explorer = RetrievalExplorer(
    retriever,
    selected_hits=selected_hits,
    default_query="",
    default_library="Stdlib",
    default_kind="start_theorem_proof",
    default_k=8,
)
retrieval_explorer.display()


After adding `exp_plus`, save this small technical context. It will be reused for `F2`, `F4`, and `F6`.


Safety net: this cell loads the expected technical context. It does not replace the retrieval exercise; it just restores the ingredients needed by the proof cells below.


In [ ]:
set_hits(
    selected_hits,
    sech_denominator_nonzero.as_retrieval_hit(),
    expected_hit("exp_plus", library="Stdlib", kind="start_theorem_proof"),
)
technical_derivative_hits = list(selected_hits)
print(format_retrieval_hits(technical_derivative_hits))


## 5. Three LLM strategies on `F2_derivative`

Here the LLM is an untrusted text model: it proposes Rocq commands, but Rocq decides whether they are valid. We compare three ways to ask it to finish the same proof state.

Before each experiment, we roll back to the checkpoint just after `auto_derive`. The selected context should contain `sech_denominator_nonzero` and `exp_plus`. The printed result includes input tokens, output tokens, and estimated dollar cost.


Strategy A is a single non-agentic call: the model receives the proof state and selected context, then returns one complete proof script. It is the cheapest when it works, but it gets no Rocq feedback before committing to the script.


In [ ]:
set_hits(selected_hits, *technical_derivative_hits)


In [ ]:
f2_max_attempts = 10
result_direct = None
direct_results = []

for attempt_id in range(1, f2_max_attempts + 1):
    f2_derivative.reverse("after_auto_derive")
    print(f"Direct attempt {attempt_id}/{f2_max_attempts}")
    result_direct = remote_llm.prove(
        f2_derivative,
        selected_hits=selected_hits,
        verbose=True,
        close=False,
    )
    direct_results.append(result_direct)
    print(result_direct)
    if result_direct.ok:
        break

show_usage("Strategy A total", direct_results)
assert result_direct is not None and result_direct.ok
f2_derivative.reverse("after_auto_derive")


Strategy B is still non-agentic, but with feedback. If the first complete script fails, we call the model again with the previous script and Rocq error. This usually costs more than A, but the error message often makes the next attempt much better.


In [ ]:
set_hits(selected_hits, *technical_derivative_hits)


In [ ]:
f2_derivative.reverse("after_auto_derive")


def result_ok(result):
    if isinstance(result, dict):
        return bool(result.get("ok"))
    return bool(getattr(result, "ok", False))


def result_part(result, name):
    if isinstance(result, dict):
        return result.get(name, "")
    return getattr(result, name, "")


feedback_context = ""
result_feedback = None
feedback_results = []

for round_id in range(3):
    f2_derivative.reverse("after_auto_derive")
    print(f"Feedback attempt {round_id}/2")
    result_feedback = remote_llm.prove(
        f2_derivative,
        selected_hits=selected_hits,
        extra_context=feedback_context,
        verbose=True,
        close=False,
    )
    feedback_results.append(result_feedback)
    print(result_feedback)

    if result_ok(result_feedback):
        break

    feedback_context += f"""
Previous attempt #{round_id + 1}:
{result_part(result_feedback, "script")}

Rocq feedback:
{result_part(result_feedback, "error")}
"""

show_usage("Strategy B total", feedback_results)
assert result_feedback is not None and result_ok(result_feedback)
f2_derivative.reverse("after_auto_derive")


Strategy C is agentic. Instead of returning a full script immediately, the model can call `run_tac` to try one tactic, inspect the new Rocq goals, and call `reverse` to roll back. It is usually the most reliable, but it can be the most expensive because each tool step may require another LLM request.


In [ ]:
set_hits(selected_hits, *technical_derivative_hits)


In [ ]:
f2_derivative.reverse("after_auto_derive")

result_agentic = remote_llm.prove(
    f2_derivative,
    selected_hits=selected_hits,
    verbose=True,
    tools={
        "run_tac": f2_derivative.run_tac,
        "reverse": f2_derivative.reverse,
    },
    max_tool_calls=48,
)
print(result_agentic)
show_usage("Strategy C total", result_agentic)


The costs should not be read as a leaderboard independent of the proof. A can be very cheap but brittle; B pays for retries; C pays for interaction with Rocq. In this TP, the interesting point is that feedback and tool use can turn a weak first attempt into a checked proof.


After one strategy succeeds, the proof is available as a local Rocq object. We will use it in two different ways: later as a formal lemma for the derivative of `F`, and now as an example passed through `extra_context` for similar derivative proofs.


In [ ]:
# Keep the selected formal context for derivative proofs minimal.
# The F2 proof itself will be passed as extra_context when it is useful as an example.
set_hits(selected_hits, *technical_derivative_hits)
print(format_retrieval_hits(selected_hits))


## 6. `F4_derivative`

The fourth-power proof has the same shape as `F2_derivative`. The selected formal ingredients are still only the denominator lemma and `exp_plus`.

The completed `F2_derivative` proof is useful, but not as a formal ingredient here: we pass it separately through `extra_context` as a nearby successful proof for the model to imitate.


As before, we first expose the definitions and run the derivative tactic by hand. Then participants can use any strategy from Section 5.

<details>
<summary>Expected retrieval hit</summary>

Name: `auto_derive`

```coq
Ltac auto_derive := ...
```

</details>


In [ ]:
f4_derivative = doc_analytic_integral.add_theorem("""Lemma F4_derivative (x : R) :
  is_derive F4 x ((sech (100 * x - 40)) ^ 4).""")

selected_hits.clear()
selected_hits.extend(technical_derivative_hits)

retrieval_explorer = RetrievalExplorer(
    retriever,
    selected_hits=selected_hits,
    default_query="",
    default_library="Coquelicot",
    default_kind="Ltac",
    default_k=8,
)
retrieval_explorer.display()


In [ ]:
f4_derivative.run_tac("unfold F4, A4, sech, tanh_exp.")
f4_derivative.run_tac("auto_derive.")
print(f4_derivative)

f4_derivative.checkpoint("after_auto_derive")


The cell below uses the agentic strategy again. The selected context contains the reusable denominator lemma, the exponential addition theorem, and the successful `F2_derivative` proof as an example.

In [ ]:
set_hits(
    selected_hits,
    *technical_derivative_hits,
)


In [ ]:
f4_derivative.reverse("after_auto_derive")

f4_example_context = f"""
Here is a closely related successful proof. Use it as an example of structure,
while adapting the details to the current goal.

{f2_derivative.source(close=True)}
"""

result = remote_llm.prove(
    f4_derivative,
    selected_hits=selected_hits,
    verbose=True,
    tools={
        "run_tac": f4_derivative.run_tac,
        "reverse": f4_derivative.reverse,
    },
    max_tool_calls=48,
    extra_context=f4_example_context,
)
print(result)
show_usage("F4_derivative", result)


In [ ]:
# The theorem is proved and can be reused later.
# For the next derivative proof, it will be passed as extra_context, not as a selected hit.
print("F4_derivative proved:")
print(f4_derivative.header)


## 7. `F6_derivative`

The sixth-power proof is again the same pattern, with more algebra and more denominator side conditions. The selected formal ingredients remain the denominator lemma and `exp_plus`.

Now we pass two nearby examples through `extra_context`: the successful `F2_derivative` and `F4_derivative` proofs. The point is to separate formal ingredients from examples of proof style.


Again, we first run `unfold` and `auto_derive` manually so the proof state is visible. After that, participants can reuse any of the strategies from Section 5.


In [ ]:
f6_derivative = doc_analytic_integral.add_theorem("""Lemma F6_derivative (x : R) :
  is_derive F6 x ((sech (1000 * x - 600)) ^ 6).""")

f6_derivative.run_tac("unfold F6, A6, sech, tanh_exp.")
f6_derivative.run_tac("auto_derive.")
print(f6_derivative)

f6_derivative.checkpoint("after_auto_derive")


Safety net: load the derivative context for `F6`. The last two hits are examples from the proofs already completed in the notebook.


In [ ]:
set_hits(
    selected_hits,
    *technical_derivative_hits,
)


In [ ]:
f6_derivative.reverse("after_auto_derive")

f6_example_context = f"""
Here are nearby successful derivative proofs. Use them as examples of structure,
while adapting the details to the current goal.

{f2_derivative.source(close=True)}

{f4_derivative.source(close=True)}
"""

result = remote_llm.prove(
    f6_derivative,
    selected_hits=selected_hits,
    verbose=True,
    tools={
        "run_tac": f6_derivative.run_tac,
        "reverse": f6_derivative.reverse,
    },
    max_tool_calls=48,
    extra_context=f6_example_context,
)
print(result)
show_usage("F6_derivative", result)


In [ ]:
selected_hits.append(f6_derivative.as_retrieval_hit())

## 8. Combining the derivative lemmas

Now we need to prove that the derivative of the sum `F2 + F4 + F6` is the sum of the derivatives. The formal ingredient to retrieve is the Coquelicot rule for the derivative of a sum.

<details>
<summary>Hint</summary>

Try a query like: `derivative of the sum of two real functions`.

</details>

<details>
<summary>Expected retrieval hit</summary>

Name: `is_derive_plus`

Signature:

```coq
is_derive_plus :
  forall (f g : R -> R) (x df dg : R),
    is_derive f x df ->
    is_derive g x dg ->
    is_derive (fun y => f y + g y) x (df + dg)
```

</details>


In [ ]:
set_hits(
    selected_hits,
    f2_derivative.as_retrieval_hit(),
    f4_derivative.as_retrieval_hit(),
    f6_derivative.as_retrieval_hit(),
)

retrieval_explorer = RetrievalExplorer(
    retriever,
    selected_hits=selected_hits,
    default_query="",
    default_library="Coquelicot",
    default_kind="start_theorem_proof",
    default_k=8,
)
retrieval_explorer.display()


In [ ]:
f_derivative = doc_analytic_integral.add_theorem("""Lemma F_derivative (x : R) :
  is_derive F x (f x).""")


Safety net: load the sum-derivative rule together with the three derivative lemmas proved above.


In [ ]:
set_hits(
    selected_hits,
    f2_derivative.as_retrieval_hit(),
    f4_derivative.as_retrieval_hit(),
    f6_derivative.as_retrieval_hit(),
    expected_hit("is_derive_plus", library="Coquelicot", kind="start_theorem_proof"),
)


In [ ]:
result = remote_llm.prove(
    f_derivative,
    selected_hits=selected_hits,
    verbose=True,
    tools={
        "run_tac": f_derivative.run_tac,
        "reverse": f_derivative.reverse,
    },
    max_tool_calls=40,
    extra_context="""Use the derivative rule for sums together with the derivative lemmas already proved.""",
)
print(result)
show_usage("F_derivative", result)


## 9. What the integral theorem asks for

The closed-form theorem uses the fundamental theorem of calculus: `F` is an antiderivative of `f`, so the integral should be `F 1 - F 0`.

There is one formal wrinkle. The theorem of calculus gives a predicate statement `is_RInt f 0 1 v`, while our definition is the real number `I := RInt f 0 1`. We therefore need two different ingredients.


First, search for the theorem that turns an antiderivative plus continuity into an `is_RInt` statement.

<details>
<summary>Hint</summary>

Try a query like: `fundamental theorem calculus real integral derivative interval is_RInt`.

</details>

<details>
<summary>Expected retrieval hit</summary>

Name: `is_RInt_derive`

```coq
is_RInt_derive :
  forall (f g : R -> R) (a b : R),
    (forall x, Rmin a b <= x <= Rmax a b -> is_derive f x (g x)) ->
    (forall x, Rmin a b <= x <= Rmax a b -> continuous g x) ->
    is_RInt g a b (f b - f a)
```

</details>


In [ ]:
integral_hits = []

retrieval_explorer = RetrievalExplorer(
    retriever,
    selected_hits=integral_hits,
    default_query="",
    default_library="Coquelicot",
    default_kind="start_theorem_proof",
    default_k=8,
)
retrieval_explorer.display()


The previous theorem gives a predicate statement. Since our goal is an equality about `RInt`, we also need the uniqueness bridge from the predicate form to the function value.

<details>
<summary>Hint for the uniqueness bridge</summary>

Try a query like: `RInt value unique is_RInt equality`.

</details>

<details>
<summary>Expected retrieval hit</summary>

Name: `is_RInt_unique`

Signature:

```coq
is_RInt_unique : forall f a b If, is_RInt f a b If -> RInt f a b = If
```

</details>


In [ ]:
retrieval_explorer = RetrievalExplorer(
    retriever,
    selected_hits=integral_hits,
    default_query="",
    default_library="Coquelicot",
    default_kind="start_theorem_proof",
    default_k=8,
)
retrieval_explorer.display()


The final theorem now has two obligations. The antiderivative obligation is handled by `F_derivative`; the missing part is continuity of `f` on the interval.


## 10. The missing continuity lemmas

We prove continuity in two steps: first show that `f` is differentiable, then use the theorem saying that differentiability implies continuity.


For the differentiability lemma, use the denominator lemma and the derivative-computation tactic. The proof starts by unfolding `f` and `sech`.

<details>
<summary>Hint</summary>

Try a query like: `automatic tactic prove derivative exists real function Coquelicot`.

</details>

<details>
<summary>Expected retrieval hit</summary>

Name: `auto_derive`

```coq
Ltac auto_derive := ...
```

</details>


In [ ]:
selected_hits.clear()
selected_hits.append(sech_denominator_nonzero.as_retrieval_hit())

retrieval_explorer = RetrievalExplorer(
    retriever,
    selected_hits=selected_hits,
    default_query="",
    default_library="Coquelicot",
    default_kind="Ltac",
    default_k=8,
)
retrieval_explorer.display()

In [ ]:
f_ex_derive = doc_analytic_integral.add_theorem("""Lemma f_ex_derive (x : R) :
  ex_derive f x.""")


Safety net: load the derivative-existence tactic and the reusable denominator lemma.


In [ ]:
set_hits(
    selected_hits,
    sech_denominator_nonzero.as_retrieval_hit(),
    expected_hit("auto_derive", library="Coquelicot", kind="ltac"),
)


In [ ]:
result = remote_llm.prove(
    f_ex_derive,
    selected_hits=selected_hits,
    verbose=True,
    tools={
        "run_tac": f_ex_derive.run_tac,
        "reverse": f_ex_derive.reverse,
    },
    max_tool_calls=36,
)
print(result)
show_usage("f_ex_derive", result)


Now we turn differentiability into continuity. The formal ingredient is the general theorem saying that a differentiable real function is continuous at the point.

<details>
<summary>Hint for the continuity lemma</summary>

Try a query like: `differentiable real function is continuous`.

</details>

<details>
<summary>Expected retrieval hit for the continuity lemma</summary>

Name: `ex_derive_continuous`

Signature:

```coq
ex_derive_continuous :
  forall (f : R -> R) (x : R), ex_derive f x -> continuous f x
```

</details>


In [ ]:
selected_hits.clear()
selected_hits.append(f_ex_derive.as_retrieval_hit())

retrieval_explorer = RetrievalExplorer(
    retriever,
    selected_hits=selected_hits,
    default_query="",
    default_library="Coquelicot",
    default_kind="start_theorem_proof",
    default_k=8,
)
retrieval_explorer.display()


In [ ]:
f_continuous = doc_analytic_integral.add_theorem("""Lemma f_continuous (x : R) :
  continuous f x.""")


Safety net: load the theorem that turns differentiability into continuity, plus the differentiability lemma just proved.


In [ ]:
set_hits(
    selected_hits,
    f_ex_derive.as_retrieval_hit(),
    expected_hit("ex_derive_continuous", library="Coquelicot", kind="start_theorem_proof"),
)


In [ ]:
result = remote_llm.prove(
    f_continuous,
    selected_hits=selected_hits,
    verbose=True,
    tools={
        "run_tac": f_continuous.run_tac,
        "reverse": f_continuous.reverse,
    },
    max_tool_calls=16,
)
print(result)
show_usage("f_continuous", result)


## 11. Closed form of the integral

We can now return to the theorem that motivated the continuity detour. The final context contains:

- the integration facts collected in Section 9;
- the antiderivative fact `F_derivative`;
- the continuity fact `f_continuous`.


In [ ]:
set_hits(
    selected_hits,
    expected_hit("is_RInt_derive", library="Coquelicot", kind="start_theorem_proof"),
    expected_hit("is_RInt_unique", library="Coquelicot", kind="start_theorem_proof"),
    f_derivative.as_retrieval_hit(),
    f_continuous.as_retrieval_hit(),
)


In [ ]:
I_closed_form_correct = doc_analytic_integral.add_theorem("""Theorem I_closed_form_correct :
  I = I_closed_form.""")

result = remote_llm.prove(
    I_closed_form_correct,
    selected_hits=selected_hits,
    extra_context=(
        "Since F is an antiderivative of f on the interval and f is continuous, "
        "the integral of f from 0 to 1 is F(1) - F(0)."
    ),
    verbose=True,
    tools={
        "run_tac": I_closed_form_correct.run_tac,
        "reverse": I_closed_form_correct.reverse,
    },
    max_tool_calls=24,
)
print(result)
show_usage("I_closed_form_correct", result)


The formal result is now:

```coq
I = F 1 - F 0
```

So the numerical dispute can be checked again by evaluating a stable version of the closed form in ordinary Python. This final Python calculation is not the proof; the proof is the Rocq theorem above.


In [ ]:
import math


def A2_num(u):
    t = math.tanh(u)
    return t


def A4_num(u):
    t = math.tanh(u)
    return t - (1 / 3) * t**3


def A6_num(u):
    t = math.tanh(u)
    return t - (2 / 3) * t**3 + (1 / 5) * t**5


def F_num(x):
    return (
        A2_num(10 * x - 2) / 10
        + A4_num(100 * x - 40) / 100
        + A6_num(1000 * x - 600) / 1000
    )

F_num(1) - F_num(0)


The value is about `0.2108027355005493`, which lies inside the certified Rocq interval and explains why the earlier `0.209736...` estimate was wrong.


## 12. Source generated by the analytic document

For debugging or comparison with `integral.v`, print the source accumulated in the analytic document.


In [ ]:
print(doc_analytic_integral.source())
